# Beijing PM2.5 Forecasting  
## Notebook 06 — Baseline Regression Models

In this notebook, we establish baseline performance using simple regression models:

- Linear Regression
- Ridge Regression

Baseline models provide a reference point to evaluate whether more complex models (e.g. XGBoost, LSTM) truly add value.

In [9]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Load Time-Based Train / Validation / Test Sets

These datasets were created using chronological splitting to prevent data leakage.

In [10]:
X_train = pd.read_csv("../data/processed/X_train.csv", index_col="datetime", parse_dates=True)
y_train = pd.read_csv("../data/processed/y_train.csv", index_col="datetime", parse_dates=True).squeeze()

X_val = pd.read_csv("../data/processed/X_val.csv", index_col="datetime", parse_dates=True)
y_val = pd.read_csv("../data/processed/y_val.csv", index_col="datetime", parse_dates=True).squeeze()

X_test = pd.read_csv("../data/processed/X_test.csv", index_col="datetime", parse_dates=True)
y_test = pd.read_csv("../data/processed/y_test.csv", index_col="datetime", parse_dates=True).squeeze()

## 2. Evaluation Metrics

We evaluate models using:

- **MAE** (Mean Absolute Error)
- **RMSE** (Root Mean Squared Error)
- **R² Score**

These metrics provide complementary views of model performance.

In [13]:
# Evaluation functions
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_model(y_true, y_pred, name="Model"):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    print(f"{name} Performance:")
    print(f"  MAE : {mae:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R²  : {r2:.3f}")
    print("-" * 30)


## 3. Linear Regression Baseline

Linear Regression provides the simplest baseline. It assumes a linear relationship between features and PM2.5.

In [ ]:
# Train linear regression
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

y_val_pred_lr = lin_reg.predict(X_val)

evaluate_model(y_val, y_val_pred_lr, name="Linear Regression (Validation)")

Linear Regression (Validation) Performance:
  MAE : 5.53
  RMSE: 9.91
  R²  : 0.980
------------------------------


## 4. Ridge Regression (L2 Regularization)

Ridge Regression adds L2 regularization to:
- reduce overfitting
- stabilize coefficients
- handle correlated features

In [15]:
# Train ridge regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

y_val_pred_ridge = ridge.predict(X_val)

evaluate_model(y_val, y_val_pred_ridge, name="Ridge Regression (Validation)")


Ridge Regression (Validation) Performance:
  MAE : 5.53
  RMSE: 9.91
  R²  : 0.980
------------------------------


## 5. Baseline Model Comparison

We compare linear and ridge regression on the validation set to select the stronger baseline.

In [17]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge Regression"],
    "MAE": [
        mean_absolute_error(y_val, y_val_pred_lr),
        mean_absolute_error(y_val, y_val_pred_ridge)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val, y_val_pred_lr)),
        np.sqrt(mean_squared_error(y_val, y_val_pred_ridge))
    ],
    "R2": [
        r2_score(y_val, y_val_pred_lr),
        r2_score(y_val, y_val_pred_ridge)
    ]
})

results

,Model,MAE,RMSE,R2
0,Linear Regression,5.534981,9.906555,0.979845
1,Ridge Regression,5.534916,9.906536,0.979845


## 6. Final Evaluation on Test Set

We evaluate the best-performing baseline model on the unseen test set.

In [ ]:
# Test evaluation for ridge regression
y_test_pred = ridge.predict(X_test)

evaluate_model(y_test, y_test_pred, name="Ridge Regression (Test)")

Ridge Regression (Test) Performance:
  MAE : 6.00
  RMSE: 10.22
  R²  : 0.988
------------------------------


In [19]:
# Test evaluation for linear regression
y_test_pred = lin_reg.predict(X_test)

evaluate_model(y_test, y_test_pred, name="Linear Regression (Test)")

Linear Regression (Test) Performance:
  MAE : 6.00
  RMSE: 10.22
  R²  : 0.988
------------------------------


## Baseline Model Interpretation

Both Linear and Ridge Regression achieve **very similar performance** on the validation and test sets, indicating minimal overfitting and strong temporal generalization.

- MAE and RMSE values suggest the models capture overall PM2.5 trends well
- High R² scores (> 0.97 on validation and ~0.99 on test) indicate that a large proportion of variance in PM2.5 is explained by the engineered features
- Regularization in Ridge Regression provides no significant improvement over standard Linear Regression, suggesting the feature set is already stable

These results establish a **strong baseline**. However, linear models may still struggle with non-linear relationships and extreme pollution events, motivating the use of more advanced models.

## Save Baseline Predictions (Optional)

Saving predictions allows error analysis and comparison with future models.

In [20]:
pd.DataFrame({
    "y_true": y_test,
    "y_pred_ridge": y_test_pred
}).to_csv("../data/processed/baseline_predictions.csv")

print("Baseline predictions saved.")

Baseline predictions saved.
